# 01 - Scraping Twitter/X: Sentimen Kebijakan Pendidikan Jakarta & Banten

Notebook ini mengambil data tweet publik terkait kebijakan pendidikan dan akses pendidikan tinggi
di DKI Jakarta dan Provinsi Banten menggunakan Apify Actor `kaitoeasyapi/twitter-x-data-tweet-scraper-pay-per-result-cheapest`.

## 1. Setup & Konfigurasi

In [24]:
from apify_client import ApifyClient
from pathlib import Path
import pandas as pd
import time
import os

env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

APIFY_API_TOKEN = os.getenv("APIFY_API_TOKEN")
if not APIFY_API_TOKEN or APIFY_API_TOKEN == "PASTE_TOKEN_KAMU_DI_SINI":
    raise ValueError("Isi APIFY_API_TOKEN di file .env dulu.")

client = ApifyClient(APIFY_API_TOKEN)

def get_run_field(run_obj, camel_key, snake_key):
    if isinstance(run_obj, dict):
        return run_obj[camel_key]
    if hasattr(run_obj, "model_dump"):
        data = run_obj.model_dump(by_alias=True)
        if camel_key in data:
            return data[camel_key]
    return getattr(run_obj, snake_key)

# Actor ID untuk kaitoeasyapi
ACTOR_ID = "kaitoeasyapi/twitter-x-data-tweet-scraper-pay-per-result-cheapest"

## 2. Definisi Keyword per Wilayah

Keyword disusun berdasarkan isu pendidikan spesifik di masing-masing wilayah,
sesuai kerangka esai (PPDB, KJP untuk Jakarta; akses sekolah, BOS untuk Banten).

In [25]:
# Keyword Jakarta
keywords_jakarta = [
    "PPDB Jakarta",
    "KJP Jakarta",
    "KJMU Jakarta",
    "zonasi sekolah Jakarta",
    "sekolah negeri Jakarta",
    "biaya kuliah Jakarta",
    "UKT Jakarta",
    "beasiswa kuliah Jakarta",
]

# Keyword Banten
keywords_banten = [
    "putus sekolah Banten",
    "akses sekolah Banten",
    "akses sekolah Serang",
    "akses sekolah Pandeglang",
    "akses sekolah Lebak",
    "BOS Banten",
    "beasiswa Banten",
    "beasiswa kuliah Banten",
    "biaya kuliah Banten",
    "PPDB Banten",
]

# Gabungkan dengan label wilayah untuk tracking
all_queries = (
    [(kw, "Jakarta") for kw in keywords_jakarta] +
    [(kw, "Banten") for kw in keywords_banten]
)

print(f"Total {len(all_queries)} query akan dijalankan:")
for kw, region in all_queries:
    print(f"  - [{region}] {kw}")

Total 18 query akan dijalankan:
  - [Jakarta] PPDB Jakarta
  - [Jakarta] KJP Jakarta
  - [Jakarta] KJMU Jakarta
  - [Jakarta] zonasi sekolah Jakarta
  - [Jakarta] sekolah negeri Jakarta
  - [Jakarta] biaya kuliah Jakarta
  - [Jakarta] UKT Jakarta
  - [Jakarta] beasiswa kuliah Jakarta
  - [Banten] putus sekolah Banten
  - [Banten] akses sekolah Banten
  - [Banten] akses sekolah Serang
  - [Banten] akses sekolah Pandeglang
  - [Banten] akses sekolah Lebak
  - [Banten] BOS Banten
  - [Banten] beasiswa Banten
  - [Banten] beasiswa kuliah Banten
  - [Banten] biaya kuliah Banten
  - [Banten] PPDB Banten


## 3. Test Run

Test dengan 1 query dan limit kecil dulu untuk memastikan format input Actor benar dan menghindari biaya tidak perlu kalau ada kesalahan konfigurasi.

In [26]:
# Test dengan 1 keyword, limit 10 tweet saja
test_input = {
    "searchTerms": ["PPDB Jakarta"],
    "maxItems": 10,
    "queryType": "Latest",  # atau "Top" jika hasil terlalu sedikit
    "lang": "in",
}

print("Menjalankan test run...")
run = client.actor(ACTOR_ID).call(run_input=test_input)
test_dataset_id = get_run_field(run, "defaultDatasetId", "default_dataset_id")

print(f"Run selesai. Status: {get_run_field(run, 'status', 'status')}")
print(f"Dataset ID: {test_dataset_id}")

Menjalankan test run...


KeyboardInterrupt: 

In [ ]:
# Lihat hasil test
test_items = list(client.dataset(test_dataset_id).iterate_items())
print(f"Jumlah tweet didapat: {len(test_items)}")

if len(test_items) > 0:
    print("\nContoh field yang tersedia:")
    print(list(test_items[0].keys()))
    print("\nContoh isi tweet pertama:")
    print(test_items[0])
else:
    print("⚠️ Tidak ada hasil. Cek kembali format input atau coba queryType='Top'")

Jumlah tweet didapat: 20

Contoh field yang tersedia:
['type', 'id', 'url', 'twitterUrl', 'text', 'source', 'retweetCount', 'replyCount', 'likeCount', 'quoteCount', 'viewCount', 'createdAt', 'lang', 'bookmarkCount', 'isReply', 'inReplyToId', 'conversationId', 'inReplyToUserId', 'inReplyToUsername', 'isPinned', 'author', 'extendedEntities', 'card', 'place', 'entities', 'reply_to_user_results', 'quoted_tweet_results', 'quoted_tweet', 'retweeted_tweet', 'isConversationControlled', 'searchTermIndex']

Contoh isi tweet pertama:
{'type': 'tweet', 'id': '2067019514299162906', 'url': 'https://x.com/SINDOnews/status/2067019514299162906', 'twitterUrl': 'https://twitter.com/SINDOnews/status/2067019514299162906', 'text': 'Pemerintah Provinsi DKI Jakarta menyiapkan 245.980 kuota murid baru dalam pelaksanaan SPMB Tahun Ajaran 2026/2027. Kuota tersebut mencakup sekolah negeri, program SPMB Bersama, hingga Sekolah Swasta Gratis. \n\nLangkah ini dilakukan untuk memperluas akses pendidikan yang lebih me

## 4. Full Scraping — Semua Keyword

Setelah test berhasil dan kamu yakin field-nya sesuai, lanjut ke scraping penuh.

**Estimasi biaya:** dengan 18 keyword pada tarif $0.18/1000 tweet:
- `maxItems=300` -> maksimal ~5.400 tweet -> sekitar **$0.97**
- `maxItems=500` -> maksimal ~9.000 tweet -> sekitar **$1.62**
- `maxItems=1000` -> maksimal ~18.000 tweet -> sekitar **$3.24**
- `maxItems=1200` -> maksimal ~21.600 tweet -> sekitar **$3.89**

Sesuaikan `MAX_ITEMS_PER_QUERY` sesuai budget kamu.

In [ ]:
MAX_ITEMS_PER_QUERY = 1200  # 18 keyword x 1200 = maksimal 21.600 tweet (~$3.89)

all_results = []

for keyword, region in all_queries:
    print(f"\nScraping: [{region}] '{keyword}' ...")
    
    run_input = {
        "searchTerms": [keyword],
        "maxItems": MAX_ITEMS_PER_QUERY,
        "queryType": "Latest",
        "lang": "in",
    }
    
    try:
        run = client.actor(ACTOR_ID).call(run_input=run_input)
        dataset_id = get_run_field(run, "defaultDatasetId", "default_dataset_id")
        items = list(client.dataset(dataset_id).iterate_items())
        
        # Tambahkan metadata keyword & region ke setiap tweet
        for item in items:
            item["_search_keyword"] = keyword
            item["_region"] = region
        
        all_results.extend(items)
        print(f"  -> Didapat {len(items)} tweet")
        
    except Exception as e:
        print(f"  -> ERROR: {e}")
        continue
    
    # Delay sopan antar request
    time.sleep(3)

print(f"\n=== TOTAL: {len(all_results)} tweet terkumpul ===")


Scraping: [Jakarta] 'PPDB Jakarta' ...


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:UAlStsKl5CXNnGv7X] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:UAlStsKl5CXNnGv7X] -> 2026-06-18T00:08:19.140Z ACTOR: Pulling container image of build FsAg9tvxqxfok3E4f from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:UAlStsKl5CXNnGv7X] -> 2026-06-18T00:08:19.143Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:UAlStsKl5CXNnGv7X] -> 2026-06-18T00:08:19.187Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:UAlStsKl5CXNnGv7X] -> 2026-06-18T00:08:19.187Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:UAlStsKl5CXNnGv7X] -> 2026-06-18T00:08:25.550Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  -> Didapat 1200 tweet

Scraping: [Jakarta] 'KJP Jakarta' ...


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Yu5kciaESRvcM1iA2] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Yu5kciaESRvcM1iA2] -> 2026-06-18T00:12:35.644Z ACTOR: Pulling container image of build FsAg9tvxqxfok3E4f from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Yu5kciaESRvcM1iA2] -> 2026-06-18T00:12:35.646Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Yu5kciaESRvcM1iA2] -> 2026-06-18T00:12:35.775Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Yu5kciaESRvcM1iA2] -> 2026-06-18T00:12:35.776Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:Yu5kciaESRvcM1iA2] -> 2026-06-18T00:12:43.188Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  -> Didapat 1200 tweet

Scraping: [Jakarta] 'KJMU Jakarta' ...


...

  -> Didapat 30 tweet

Scraping: [Banten] 'biaya kuliah Banten' ...


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:hzxlw5xSwvgwXXqzf] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:hzxlw5xSwvgwXXqzf] -> 2026-06-18T00:38:32.689Z ACTOR: Pulling container image of build FsAg9tvxqxfok3E4f from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:hzxlw5xSwvgwXXqzf] -> 2026-06-18T00:38:32.692Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:hzxlw5xSwvgwXXqzf] -> 2026-06-18T00:38:32.771Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:hzxlw5xSwvgwXXqzf] -> 2026-06-18T00:38:32.772Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:hzxlw5xSwvgwXXqzf] -> 2026-06-18T00:38:40.172Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  -> Didapat 28 tweet

Scraping: [Banten] 'PPDB Banten' ...


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nhVtWsVsaayw5nWKl] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nhVtWsVsaayw5nWKl] -> 2026-06-18T00:39:04.731Z ACTOR: Pulling container image of build FsAg9tvxqxfok3E4f from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nhVtWsVsaayw5nWKl] -> 2026-06-18T00:39:04.733Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nhVtWsVsaayw5nWKl] -> 2026-06-18T00:39:04.772Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nhVtWsVsaayw5nWKl] -> 2026-06-18T00:39:04.772Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:nhVtWsVsaayw5nWKl] -> 2026-06-18T00:39:11.741Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  -> Didapat 661 tweet

=== TOTAL: 7870 tweet terkumpul ===


## 5. Simpan ke DataFrame & CSV

In [ ]:
df = pd.DataFrame(all_results)

if df.empty:
    raise ValueError("Tidak ada tweet terkumpul. Cek keyword, token, atau ubah queryType ke 'Top'.")

# Pecah metadata author agar mudah dicek tanpa error karena kolom author berisi dict.
if "author" in df.columns:
    df["author_username"] = df["author"].apply(lambda x: x.get("userName") if isinstance(x, dict) else None)
    df["author_name"] = df["author"].apply(lambda x: x.get("name") if isinstance(x, dict) else None)
    df["author_followers"] = df["author"].apply(lambda x: x.get("followers") if isinstance(x, dict) else None)

print(f"Shape: {df.shape}")
print(f"\nDistribusi per wilayah:")
print(df["_region"].value_counts())
print(f"\nDistribusi per keyword:")
print(df["_search_keyword"].value_counts())

df.head()

Shape: (7870, 37)

Distribusi per wilayah:
_region
Jakarta    6413
Banten     1457
Name: count, dtype: int64

Distribusi per keyword:
_search_keyword
PPDB Jakarta                1200
KJP Jakarta                 1200
sekolah negeri Jakarta      1200
KJMU Jakarta                1168
PPDB Banten                  661
UKT Jakarta                  541
biaya kuliah Jakarta         441
BOS Banten                   420
beasiswa kuliah Jakarta      377
zonasi sekolah Jakarta       286
beasiswa Banten              186
putus sekolah Banten          47
akses sekolah Banten          36
beasiswa kuliah Banten        30
biaya kuliah Banten           28
akses sekolah Pandeglang      19
akses sekolah Serang          15
akses sekolah Lebak           15
Name: count, dtype: int64


,type,id,url,twitterUrl,text,source,retweetCount,replyCount,likeCount,quoteCount,...,quoted_tweet,retweeted_tweet,isConversationControlled,searchTermIndex,_search_keyword,_region,isQuote,author_username,author_name,author_followers
0,tweet,2067019514299162906,https://x.com/SINDOnews/status/206701951429916...,https://twitter.com/SINDOnews/status/206701951...,Pemerintah Provinsi DKI Jakarta menyiapkan 245...,,0.0,0.0,1.0,0.0,...,None,NaN,False,0.0,PPDB Jakarta,Jakarta,NaN,SINDOnews,SINDOnews,401924.0
1,tweet,2066140048022761832,https://x.com/megncheesee/status/2066140048022...,https://twitter.com/megncheesee/status/2066140...,"Capek banget ngikutinnya, aku udah pernah rasa...",,0.0,0.0,0.0,0.0,...,None,NaN,False,0.0,PPDB Jakarta,Jakarta,NaN,megncheesee,͗͏Meg🅰️ slr,101.0
2,tweet,2065258641356439929,https://x.com/cancoi/status/2065258641356439929,https://twitter.com/cancoi/status/206525864135...,LOWONGAN PEKERJAAN SD Islam Al Ikhlas Cipete\n...,,0.0,0.0,0.0,0.0,...,None,NaN,False,0.0,PPDB Jakarta,Jakarta,NaN,cancoi,Puput,1359.0
3,tweet,2064626708259750008,https://x.com/another_io/status/20646267082597...,https://twitter.com/another_io/status/20646267...,"ga pcmb jabar, ga ppdb jakarta, sistem penerim...",,0.0,0.0,0.0,0.0,...,None,NaN,False,0.0,PPDB Jakarta,Jakarta,NaN,another_io,io,141.0
4,tweet,2064230070198456579,https://x.com/ccaraela/status/2064230070198456579,https://twitter.com/ccaraela/status/2064230070...,the things that aku alhamdulillahkan adalah ak...,,0.0,1.0,0.0,0.0,...,None,NaN,False,0.0,PPDB Jakarta,Jakarta,NaN,ccaraela,daveenaaaaaa 10 + 6,20.0


In [ ]:
# Cek duplikasi sebelum simpan (penting untuk tahap filter bot nanti)
id_col = "id" if "id" in df.columns else None
n_duplicate = df.duplicated(subset=[id_col]).sum() if id_col else 0
print(f"Jumlah duplikat terdeteksi: {n_duplicate}")

df_clean = df.drop_duplicates(subset=[id_col]) if id_col else df.copy()
print(f"Total setelah dedup: {len(df_clean)}")

# Simpan raw data (sebelum filter bot/spam - itu tahap berikutnya)
output_path = "raw_tweets_jakarta_banten.csv"
df_clean.to_csv(output_path, index=False)
print(f"\nData disimpan ke: {output_path}")

Jumlah duplikat terdeteksi: 614
Total setelah dedup: 7256

Data disimpan ke: raw_tweets_jakarta_banten.csv


## 6. Cek Cepat Distribusi Data

Sebelum lanjut ke notebook berikutnya (filter bot & spam), cek dulu kualitas data mentah:
apakah ada akun yang muncul terlalu sering (indikasi bot), atau isi tweet yang sangat mirip.

In [ ]:
# Cari kolom username (nama field bisa beda-beda tergantung Actor)
preferred_username_cols = ["author_username", "username", "userName", "screen_name"]
username_candidates = [c for c in preferred_username_cols if c in df_clean.columns]
print(f"Kolom kandidat username: {username_candidates}")

if username_candidates:
    col = username_candidates[0]
    print(f"\nTop 10 akun paling sering muncul (kolom: {col}):")
    print(df_clean[col].value_counts().head(10))
    print("\n⚠️ Akun dengan frekuensi sangat tinggi perlu diperiksa di tahap filter bot selanjutnya.")

Kolom kandidat username: ['author_username']

Top 10 akun paling sering muncul (kolom: author_username):
author_username
kompascom         431
schfess           317
DKIJakarta        265
detikcom          257
RadioElshinta     162
tempodotco        143
mediaindonesia    121
tvOneNews         103
SchoolfessID      100
KompasTV           96
Name: count, dtype: int64

⚠️ Akun dengan frekuensi sangat tinggi perlu diperiksa di tahap filter bot selanjutnya.


## Langkah Selanjutnya

1. Lanjut ke `02_bot_filter.ipynb` untuk Data Verification Layer (filter spam, bot, CIB)
2. Cek manual beberapa sample tweet untuk memastikan relevansi dengan topik pendidikan
3. Jika hasil terlalu sedikit per keyword, coba ubah `queryType` ke `"Top"` atau perluas keyword